# Tutorial Assignment: LU 4 - Kalman Filter Position Estimation

### Aim

The aim of this tutorial is to implement a Kalman filter to estimate the position of a person using accelerometer measurements recorded from a mobile phone.

The phone movement is modelled in two dimensions. The first part of the journey is a horizontal walk from the ground floor of building N1A to the lift. The second part of the journey is the lift movement to the top floor.

The Kalman filter is implemented twice:

1. Using only the measured accelerometer signal as the system input.
2. Using the measured accelerometer signal together with an inferred velocity signal.

## 1. Problem Analysis

The task is to estimate the position of a phone carried by a person. The phone accelerometer provides acceleration measurements, but position is not measured directly. Position must therefore be estimated from acceleration.

This is difficult because position is obtained by integrating acceleration twice. Any small acceleration bias or sensor noise can therefore grow into a large position error over time.

The route consists of two main stages:

1. A straight horizontal walk of approximately 5 m to 10 m toward the lift.
2. A lift movement to the top floor.

For this tutorial, the true route is known beforehand. This route is used as the reference position. The elapsed time and known distance are also used to infer an approximate velocity signal.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.interpolate import interp1d
from scipy.signal import butter, filtfilt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

## 2. Load the Accelerometer Data

The accelerometer data recorded from the mobile phone is loaded from a CSV file. The important columns are the elapsed time and the two acceleration axes that were approximately parallel to the ground.

The phone sensors may not sample at a perfectly constant rate, so the time column must be inspected before the Kalman filter is applied.

In [ ]:
file_path = "Raw Data.csv"
data = pd.read_csv(file_path)
data.head()

In [ ]:
data.info()
data.describe()

In [ ]:
print(data.columns)

In [ ]:
t_raw = data["Time (s)"].to_numpy()
ax_raw = data["Linear Acceleration x (m/s^2)"].to_numpy()
ay_raw = data["Linear Acceleration y (m/s^2)"].to_numpy()
az_raw = data["Linear Acceleration z (m/s^2)"].to_numpy()
a_abs_raw = data["Absolute acceleration (m/s^2)"].to_numpy()

# Shift time so that it starts at zero
t_raw = t_raw - t_raw[0]

print("Number of samples:", len(t_raw))
print("Total recording time:", t_raw[-1], "s")
print("Average sample time:", np.mean(np.diff(t_raw)), "s")
print("Approximate sample frequency:", 1 / np.mean(np.diff(t_raw)), "Hz")
print("Minimum dt:", np.min(np.diff(t_raw)))
print("Maximum dt:", np.max(np.diff(t_raw)))

## 3. Raw Accelerometer Signal Analysis

The accelerometer data contains 3043 samples over a recording time of approximately 30.5 seconds. The data contains linear acceleration in the x, y and z directions, as well as the absolute acceleration magnitude.

Before implementing the Kalman filter, the raw signal must be analysed to identify which parts of the signal represent useful movement and which parts are disturbances.

The useful movement sections are expected to be:

1. The walking section, where the horizontal acceleration changes due to the person walking toward the lift.
2. The lift section, where the acceleration changes mainly when the lift starts and stops.

The disturbances in the signal include sensor noise, hand vibration, step impacts, phone tilt and small acceleration bias. These disturbances are important because position is obtained by double integration of acceleration, meaning that even small acceleration errors can grow into large position errors.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(t_raw, ax_raw, label="x acceleration")
plt.plot(t_raw, ay_raw, label="y acceleration")
plt.plot(t_raw, az_raw, label="z acceleration")
plt.plot(t_raw, a_abs_raw, label="absolute acceleration", alpha=0.6)

plt.xlabel("Time [s]")
plt.ylabel("Acceleration [m/s²]")
plt.title("Raw accelerometer signals")
plt.legend()
plt.grid(True)
plt.show()

## 4. Identification of Important Signal Sections

The raw acceleration plot shows that the recording can be divided into separate movement stages.

The first stage occurs from approximately 0 s to 13 s. This section contains large oscillations, especially in the horizontal acceleration signals. These oscillations are caused by walking, where each step produces a short acceleration impulse.

From approximately 14 s to 18 s, the acceleration signal is much smaller and remains close to zero. This section is treated as the stationary calibration section. It is useful for estimating and removing the accelerometer bias.

The lift movement occurs from approximately 19 s to 29 s. The first smaller bump around 20 s indicates the lift starting to move, while the disturbance around 27 s to 28 s indicates the lift slowing down and stopping. Between these points, the lift moves more smoothly and the acceleration remains relatively small.

The important parts of the signal are the walking section and the lift movement section. The disturbances include step impacts, hand vibration, phone tilt, sensor noise and small bias errors.

In [ ]:
T_WALK_START = 0.0
T_WALK_END = 13.0

T_CAL_START = 14.0
T_CAL_END = 18.0

T_LIFT_START = 19.0
T_LIFT_END = 29.0

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(t_raw, ax_raw, label="x acceleration")
plt.plot(t_raw, ay_raw, label="y acceleration")
plt.plot(t_raw, az_raw, label="z acceleration")
plt.plot(t_raw, a_abs_raw, label="absolute acceleration", alpha=0.6)

plt.axvline(T_WALK_END, color="red", linestyle="--", label="End of walk")
plt.axvline(T_CAL_START, color="orange", linestyle="--", label="Calibration start")
plt.axvline(T_CAL_END, color="purple", linestyle="--", label="Calibration end")
plt.axvline(T_LIFT_START, color="green", linestyle="--", label="Lift start")
plt.axvline(T_LIFT_END, color="black", linestyle="--", label="Lift end")

plt.xlabel("Time [s]")
plt.ylabel("Acceleration [m/s²]")
plt.title("Raw accelerometer signal with identified movement stages")
plt.legend()
plt.grid(True)
plt.show()

## 5. Preprocessing

The raw accelerometer signal contains useful motion information, but it also contains unwanted disturbances. These disturbances include sensor noise, small bias errors, hand vibration, phone tilt and high-frequency movement caused by walking steps.

Before the Kalman filter is applied, the signal is preprocessed to make it more suitable for position estimation.

The following preprocessing steps are applied:

1. **Resampling:** The phone sampled at approximately 100 Hz. The data is resampled to 20 Hz to reduce the number of samples and smooth small sampling irregularities.

2. **Bias removal:** A stationary section from 14 s to 18 s is used to estimate the accelerometer bias. This bias is subtracted from the full signal.

3. **Low-pass filtering:** A Butterworth low-pass filter is applied to reduce high-frequency disturbances caused by steps, hand movement and sensor noise.

These steps are necessary because position is obtained from acceleration by integration. If the acceleration signal contains bias or high-frequency noise, the resulting position estimate can drift significantly.

In [ ]:
# Resample the signal to a lower fixed sampling frequency

dt = 0.05 # 20 Hz sampling period
fs = 1 / dt # sampling frequency

t = np.arange(t_raw[0], t_raw[-1], dt)

# Interpolation functions
fx = interp1d(t_raw, ax_raw, fill_value="extrapolate")
fy = interp1d(t_raw, ay_raw, fill_value="extrapolate")
fz = interp1d(t_raw, az_raw, fill_value="extrapolate")
fabs = interp1d(t_raw, a_abs_raw, fill_value="extrapolate")

# Resampled signals
ax = fx(t)
ay = fy(t)
az = fz(t)
a_abs = fabs(t)

print("Original number of samples:", len(t_raw))
print("Resampled number of samples:", len(t))
print("Original approximate sampling frequency:", 1 / np.mean(np.diff(t_raw)), "Hz")
print("New sampling frequency:", fs, "Hz")
print("New sampling time dt:", dt, "s")

### 5.1 Resampling

The original accelerometer signal was sampled at approximately 100 Hz. Although this is a good sampling rate, it results in many samples and may still contain small timing variations.

The signal was resampled to 20 Hz using linear interpolation. This gives a fixed sampling interval of:

$$
\Delta t = 0.05 \text{ s}
$$

A fixed sampling interval is required because the Kalman filter matrices depend on the value of $\Delta t$. Downsampling also helps reduce small high-frequency variations in the signal before the Kalman filter is applied.

In [ ]:
# Bias removal using the stationary calibration section

cal_mask = (t >= T_CAL_START) & (t <= T_CAL_END)

bias_x = np.mean(ax[cal_mask])
bias_y = np.mean(ay[cal_mask])
bias_z = np.mean(az[cal_mask])

ax_detrended = ax - bias_x
ay_detrended = ay - bias_y
az_detrended = az - bias_z

print("Estimated x-axis bias:", bias_x)
print("Estimated y-axis bias:", bias_y)
print("Estimated z-axis bias:", bias_z)

### 5.2 Bias Removal

The stationary section from 14 s to 18 s was used to estimate the accelerometer bias. During this section, the phone was assumed to be approximately stationary, so the acceleration should ideally be close to zero.

The mean acceleration in this section was calculated for each axis and subtracted from the full signal.

This is important because even a small constant acceleration error can cause a large position error after double integration.

In [ ]:
# Low-pass Butterworth filter

def lowpass_filter(signal, cutoff, fs, order=4):
 nyquist = 0.5 * fs
 normal_cutoff = cutoff / nyquist
 
 b, a = butter(order, normal_cutoff, btype="low")
 filtered_signal = filtfilt(b, a, signal)
 
 return filtered_signal

cutoff = 2.0 # Hz

ax_filt = lowpass_filter(ax_detrended, cutoff, fs)
ay_filt = lowpass_filter(ay_detrended, cutoff, fs)
az_filt = lowpass_filter(az_detrended, cutoff, fs)

print("Low-pass filter cutoff frequency:", cutoff, "Hz")

### 5.3 Low-pass Filtering

A low-pass Butterworth filter was applied to the detrended acceleration signals.

The cutoff frequency was chosen as 2 Hz. This keeps the slower body and lift motion while reducing the high-frequency disturbances caused by step impacts, hand vibration and sensor noise.

This filtering step helps prevent high-frequency acceleration noise from being amplified during integration.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(t, ax_detrended, label="x detrended", alpha=0.4)
plt.plot(t, ax_filt, label="x filtered", linewidth=2)
plt.xlabel("Time [s]")
plt.ylabel("Acceleration [m/s²]")
plt.title("Preprocessing result for x-axis acceleration")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(t, ay_detrended, label="y detrended", alpha=0.4)
plt.plot(t, ay_filt, label="y filtered", linewidth=2)
plt.xlabel("Time [s]")
plt.ylabel("Acceleration [m/s²]")
plt.title("Preprocessing result for y-axis acceleration")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(t, az_detrended, label="z detrended", alpha=0.4)
plt.plot(t, az_filt, label="z filtered", linewidth=2)
plt.xlabel("Time [s]")
plt.ylabel("Acceleration [m/s²]")
plt.title("Preprocessing result for z-axis acceleration")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(t, ax_filt, label="processed x acceleration")
plt.plot(t, ay_filt, label="processed y acceleration")
plt.plot(t, az_filt, label="processed z acceleration")

plt.axvline(T_WALK_END, color="red", linestyle="--", label="End of walk")
plt.axvline(T_CAL_START, color="orange", linestyle="--", label="Calibration start")
plt.axvline(T_CAL_END, color="purple", linestyle="--", label="Calibration end")
plt.axvline(T_LIFT_START, color="green", linestyle="--", label="Lift start")
plt.axvline(T_LIFT_END, color="black", linestyle="--", label="Lift end")

plt.xlabel("Time [s]")
plt.ylabel("Acceleration [m/s²]")
plt.title("Processed accelerometer signals")
plt.legend()
plt.grid(True)
plt.show()

### 5.4 Preprocessing Discussion

The preprocessing improved the accelerometer signal by reducing the effect of sensor bias and high-frequency disturbances.

The detrended signal removes the average offset measured during the stationary calibration section. The filtered signal is smoother than the original signal, which makes it more suitable for integration and Kalman filtering.

The walking section still contains visible acceleration changes because these are caused by actual movement and step impacts. However, the very sharp high-frequency variations have been reduced.

The lift section is also smoother after filtering. The acceleration changes at the start and end of the lift are still visible, which is useful because these points represent real motion changes.

This processed acceleration signal will be used as the input to the Kalman filter model.

## 6. Actual Route and Inferred Velocity

The actual route is based on the predetermined path that was followed during the experiment.

The movement is divided into two main sections:

1. A horizontal walking section from the starting point to the lift.
2. A vertical lift section from the ground floor to the top floor.

The actual position is approximated using the known distances and the measured time intervals from the accelerometer signal.

The inferred velocity is calculated using:

$$
v = \frac{d}{t}
$$

where $d$ is the known distance and $t$ is the time taken to complete that section of the journey.

The inferred velocity is later used in the second Kalman filter implementation to improve the position estimate.

In [ ]:
# Known route distances
walk_distance = 6.0
lift_distance = 11.0

# Time durations from the identified signal sections
walk_duration = T_WALK_END - T_WALK_START
lift_duration = T_LIFT_END - T_LIFT_START

# Inferred average velocities
v_walk = walk_distance / walk_duration
v_lift = lift_distance / lift_duration

print("Walking duration:", walk_duration, "s")
print("Lift duration:", lift_duration, "s")
print("Inferred walking velocity:", v_walk, "m/s")
print("Inferred lift velocity:", v_lift, "m/s")

In [ ]:
def smoothstep(s):
 """
 Smooth transition function.
 s must be between 0 and 1.
 The output starts and ends with zero slope.
 """
 return 3*s**2 - 2*s**3


def smooth_position(t, t_start, t_end, distance):
 """
 Creates a smooth position curve from 0 to distance.
 The motion starts and ends smoothly.
 """
 pos = np.zeros_like(t)
 
 for k, tk in enumerate(t):
 if tk < t_start:
 pos[k] = 0
 elif t_start <= tk <= t_end:
 s = (tk - t_start) / (t_end - t_start)
 pos[k] = distance * smoothstep(s)
 else:
 pos[k] = distance
 
 return pos


# Known route distances
walk_distance = 6.0 # metres walked horizontally
lift_distance = 11.0 # metres travelled vertically in lift

# Smooth actual position profiles
p_y_true = smooth_position(t, T_WALK_START, T_WALK_END, walk_distance)
p_z_true = smooth_position(t, T_LIFT_START, T_LIFT_END, lift_distance)

# Inferred velocity from the gradient of the smooth position
v_y_true = np.gradient(p_y_true, dt)
v_z_true = np.gradient(p_z_true, dt)

print("Maximum inferred walking velocity:", np.max(v_y_true), "m/s")
print("Maximum inferred lift velocity:", np.max(v_z_true), "m/s")

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(t, p_y_true, label="actual horizontal position")
plt.plot(t, p_z_true, label="actual lift position")

plt.axvline(T_WALK_END, color="red", linestyle="--", label="End of walk")
plt.axvline(T_LIFT_START, color="green", linestyle="--", label="Lift start")
plt.axvline(T_LIFT_END, color="black", linestyle="--", label="Lift end")

plt.xlabel("Time [s]")
plt.ylabel("Position [m]")
plt.title("Actual route based on measured distances")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(t, v_y_true, label="inferred horizontal velocity")
plt.plot(t, v_z_true, label="inferred lift velocity")

plt.axvline(T_WALK_END, color="red", linestyle="--", label="End of walk")
plt.axvline(T_LIFT_START, color="green", linestyle="--", label="Lift start")
plt.axvline(T_LIFT_END, color="black", linestyle="--", label="Lift end")

plt.xlabel("Time [s]")
plt.ylabel("Velocity [m/s]")
plt.title("Inferred velocity from actual route")
plt.legend()
plt.grid(True)
plt.show()

### 6.1 Route and Velocity Discussion

The actual route was approximated using the known measured distances and the time intervals identified from the accelerometer signal.

Instead of assuming that the person and lift move at a perfectly constant velocity, a smooth position profile was used. This is more realistic because a person does not instantly start walking at full speed, and a lift also accelerates and decelerates at the start and end of its motion.

A smoothstep function was used to model the position transition during the walking and lift sections. This creates a position curve with a gradual start and stop.

The inferred velocity was then calculated from the gradient of the smooth position curve. This gives a more realistic velocity estimate than a sudden step change in velocity.

The inferred velocity is still an approximation because the actual walking speed and lift speed were not measured directly. It is therefore treated as a noisy measurement in the second Kalman filter.

## 7. Kalman Filter Model

The Kalman filter is used to estimate the position and velocity of the phone during the journey.

The state vector is chosen as:

$$
x_k =
\begin{bmatrix}
p_y \\
p_z \\
v_y \\
v_z
\end{bmatrix}
$$

where:

- $p_y$ is the horizontal walking position,
- $p_z$ is the lift position,
- $v_y$ is the horizontal velocity,
- $v_z$ is the lift velocity.

The control input is the measured acceleration:

$$
u_k =
\begin{bmatrix}
a_y \\
a_z
\end{bmatrix}
$$

The $y$-axis acceleration is used for the horizontal walking direction because it showed the strongest walking motion. The $z$-axis acceleration is used for the lift direction because it showed the clearest lift start and stop behaviour.

A constant acceleration model is used because the phone provides acceleration measurements. The model is based on the following motion equations:

$$
p_{k+1} = p_k + v_k \Delta t + \frac{1}{2}a_k \Delta t^2
$$

$$
v_{k+1} = v_k + a_k \Delta t
$$

### 7.1 Kalman Filter Equations

The Kalman filter consists of a prediction step and a correction step.

Prediction:

$$
\hat{x}_{k}^{-} = A\hat{x}_{k-1} + Bu_k
$$

$$
P_k^{-} = AP_{k-1}A^T + Q
$$

Correction:

$$
K_k = P_k^{-}H^T(HP_k^{-}H^T + R)^{-1}
$$

$$
\hat{x}_k = \hat{x}_k^{-} + K_k(z_k - H\hat{x}_k^{-})
$$

$$
P_k = (I - K_kH)P_k^{-}
$$

where:

- $A$ is the state transition matrix,
- $B$ is the input matrix,
- $Q$ is the process noise covariance matrix,
- $H$ is the measurement matrix,
- $R$ is the measurement noise covariance matrix,
- $K$ is the Kalman gain,
- $P$ is the estimate covariance matrix.

In [ ]:
# Kalman filter matrices for the state vector:
# x = [p_y, p_z, v_y, v_z]^T

A = np.array([
 [1, 0, dt, 0],
 [0, 1, 0, dt],
 [0, 0, 1, 0],
 [0, 0, 0, 1]
])

B = np.array([
 [0.5 * dt**2, 0],
 [0, 0.5 * dt**2],
 [dt, 0],
 [0, dt]
])

# Process noise covariance
# Higher values mean the model is less trusted
Q = np.diag([0.05, 0.05, 0.2, 0.2])

# Initial covariance
P0 = np.eye(4) * 1.0

I = np.eye(4)

print("A matrix:")
print(A)

print("\nB matrix:")
print(B)

print("\nQ matrix:")
print(Q)

print("\nP0 matrix:")
print(P0)

### 7.2 Motivation for Matrix Choices

The matrix $A$ is selected from the constant acceleration motion model. It updates position using the previous position and velocity.

The matrix $B$ maps the measured acceleration input into the position and velocity states. The acceleration affects position through:

$$
\frac{1}{2}a\Delta t^2
$$

and velocity through:

$$
a\Delta t
$$

The process noise covariance matrix $Q$ represents uncertainty in the model. This uncertainty comes from phone tilt, hand movement, step impacts, sensor noise and the fact that human walking is not perfectly described by a simple constant acceleration model.

The initial covariance matrix $P_0$ represents the uncertainty in the starting state. The phone starts at the origin, but some uncertainty is still included so that the filter can update during the estimation process.

The measurement noise covariance matrix $R$ is chosen based on how reliable the measurement is. For Filter 1, the position measurement comes from double-integrated acceleration, so a larger measurement noise is used. For Filter 2, the measurement is the inferred velocity from the known route and elapsed time, so it is treated as more reliable than the acceleration-only position estimate.

## 8. Kalman Filter 1: Accelerometer Signal Only

In the first implementation, only the measured accelerometer signal is used.

The processed acceleration signal is used as the input to the Kalman model. Since the phone does not directly measure position, a rough position measurement is created by integrating the acceleration signal twice.

This method is expected to contain drift because any small acceleration error becomes larger after integration. This filter is therefore used to show the limitations of using only accelerometer data for position estimation.

In [ ]:
# Use y-axis as horizontal walking acceleration
# Use z-axis as lift acceleration

a_y = ay_filt
a_z = az_filt

def integrate_signal(signal, dt):
 return np.cumsum(signal) * dt

# First integration: acceleration to velocity
v_y_acc = integrate_signal(a_y, dt)
v_z_acc = integrate_signal(a_z, dt)

# Second integration: velocity to position
p_y_acc = integrate_signal(v_y_acc, dt)
p_z_acc = integrate_signal(v_z_acc, dt)

# Simple drift correction using known final distances
p_y_acc_corr = p_y_acc - np.linspace(p_y_acc[0], p_y_acc[-1] - walk_distance, len(p_y_acc))
p_z_acc_corr = p_z_acc - np.linspace(p_z_acc[0], p_z_acc[-1] - lift_distance, len(p_z_acc))

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(t, p_y_acc, label="raw double-integrated horizontal position", alpha=0.5)
plt.plot(t, p_y_acc_corr, label="drift-corrected horizontal position")
plt.plot(t, p_y_true, "k--", label="actual horizontal position")
plt.xlabel("Time [s]")
plt.ylabel("Position [m]")
plt.title("Acceleration-only horizontal position measurement")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(t, p_z_acc, label="raw double-integrated lift position", alpha=0.5)
plt.plot(t, p_z_acc_corr, label="drift-corrected lift position")
plt.plot(t, p_z_true, "k--", label="actual lift position")
plt.xlabel("Time [s]")
plt.ylabel("Position [m]")
plt.title("Acceleration-only lift position measurement")
plt.legend()
plt.grid(True)
plt.show()

### 8.1 Acceleration-Only Position Measurement Discussion

The acceleration-only position was obtained by integrating the processed acceleration signal twice. The first integration gives velocity and the second integration gives position.

The horizontal position estimate follows the general shape of the actual walking route. However, it does not perfectly match the actual route because the accelerometer signal still contains noise, phone tilt effects and walking disturbances.

The lift position estimate shows a larger difference between the raw double-integrated position and the actual lift position. This demonstrates the drift problem clearly. Small errors in the acceleration signal accumulate during integration and cause the position estimate to move away from the true route.

A simple drift correction was applied using the known final distances. This improved the acceleration-only position measurement, but it is still not a perfect measurement. This corrected position will be used as a noisy measurement in the first Kalman filter.

In [ ]:
def run_kalman(u, z, H, R, A, B, Q, P0):
 n = len(u)
 nx = A.shape[0]

 x_est = np.zeros((n, nx))
 x_pred = np.zeros((n, nx))
 K_hist = np.zeros((n, nx, H.shape[0]))

 x = np.zeros(nx)
 P = P0.copy()

 for k in range(n):
 # Prediction step
 x_minus = A @ x + B @ u[k]
 P_minus = A @ P @ A.T + Q

 # Correction step
 S = H @ P_minus @ H.T + R
 K = P_minus @ H.T @ np.linalg.inv(S)

 innovation = z[k] - H @ x_minus
 x = x_minus + K @ innovation
 P = (np.eye(nx) - K @ H) @ P_minus

 x_pred[k] = x_minus
 x_est[k] = x
 K_hist[k] = K

 return x_est, x_pred, K_hist

In [ ]:
# Control input: processed acceleration in horizontal and lift directions
u = np.column_stack((a_y, a_z))

# Measurement vector for Filter 1: acceleration-only position measurements
z1 = np.column_stack((p_y_acc_corr, p_z_acc_corr))

# Measurement matrix for position measurements only
H1 = np.array([
 [1, 0, 0, 0],
 [0, 1, 0, 0]
])

# Measurement noise covariance
# These values are relatively large because double-integrated acceleration is noisy
R1 = np.diag([2.0, 2.0])

x_est_1, x_pred_1, K1 = run_kalman(u, z1, H1, R1, A, B, Q, P0)

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(t, p_y_true, "k--", label="actual horizontal position")
plt.plot(t, x_pred_1[:, 0], label="predicted horizontal position")
plt.plot(t, x_est_1[:, 0], label="estimated horizontal position")
plt.xlabel("Time [s]")
plt.ylabel("Position [m]")
plt.title("Filter 1: Horizontal position using accelerometer only")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(t, p_z_true, "k--", label="actual lift position")
plt.plot(t, x_pred_1[:, 1], label="predicted lift position")
plt.plot(t, x_est_1[:, 1], label="estimated lift position")
plt.xlabel("Time [s]")
plt.ylabel("Position [m]")
plt.title("Filter 1: Lift position using accelerometer only")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
plt.plot(p_y_true, p_z_true, "k--", label="actual route")
plt.plot(x_pred_1[:, 0], x_pred_1[:, 1], label="predicted route")
plt.plot(x_est_1[:, 0], x_est_1[:, 1], label="estimated route")

plt.xlabel("Horizontal position [m]")
plt.ylabel("Lift position [m]")
plt.title("Filter 1: Route comparison using accelerometer only")
plt.legend()
plt.grid(True)
plt.axis("equal")
plt.show()

### 8.2 Filter 1 Results and Discussion

The first Kalman filter used only the processed accelerometer signal. The acceleration was used as the model input, while the position measurement was obtained by double-integrating the acceleration signal.

The horizontal position estimate follows the general shape of the actual walking route and reaches approximately the correct final distance of 6 m. However, the estimate does not follow the true straight-line route perfectly. This is caused by step impacts, hand vibration, sensor noise and phone tilt during walking.

The lift position estimate shows a larger error. The estimated lift position starts increasing before the actual lift movement begins at 19 s. This indicates that the z-axis acceleration still contains drift after preprocessing. Since position is obtained by double integration, small acceleration errors accumulate and appear as false position changes.

The route comparison shows this behaviour clearly. The estimated route does not remain flat during the stationary period before the lift. Instead, the estimated lift position slowly increases before the actual lift movement starts.

This result shows the limitation of using only accelerometer data for position estimation. Although preprocessing improves the signal, accelerometer-only position estimation is still affected by drift.

## 9. Kalman Filter 2: Accelerometer Signal and Inferred Velocity

In the second implementation, the accelerometer signal is still used as the input to the Kalman filter model. However, an inferred velocity signal is added as a correction measurement.

The inferred velocity is calculated from the known route distance and the time taken to complete each section of the journey.

The measurement vector is:

$$
z_k =
\begin{bmatrix}
v_y \\
v_z
\end{bmatrix}
$$

where $v_y$ is the inferred horizontal walking velocity and $v_z$ is the inferred lift velocity.

The measurement matrix is chosen as:

$$
H =
\begin{bmatrix}
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

This matrix tells the Kalman filter that only the velocity states are being measured directly. The position states are still estimated through the motion model.

This approach is useful because the velocity information helps prevent the filter from drifting when the acceleration signal contains bias or noise.


In [ ]:
# Measurement vector for Filter 2:
# only inferred velocity from the known route is used as the correction measurement
z2 = np.column_stack((v_y_true, v_z_true))

# Measurement matrix:
# this maps the state vector [p_y, p_z, v_y, v_z] to the measured velocities [v_y, v_z]
H2 = np.array([
 [0, 0, 1, 0],
 [0, 0, 0, 1]
])

# Measurement noise covariance
# These values are small because the inferred velocity comes from the known route
R2 = np.diag([0.08, 0.08])

x_est_2, x_pred_2, K2 = run_kalman(u, z2, H2, R2, A, B, Q, P0)

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(t, p_y_true, "k--", label="actual horizontal position")
plt.plot(t, x_pred_2[:, 0], label="predicted horizontal position")
plt.plot(t, x_est_2[:, 0], label="estimated horizontal position")
plt.xlabel("Time [s]")
plt.ylabel("Position [m]")
plt.title("Filter 2: Horizontal position with inferred velocity")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(t, p_z_true, "k--", label="actual lift position")
plt.plot(t, x_pred_2[:, 1], label="predicted lift position")
plt.plot(t, x_est_2[:, 1], label="estimated lift position")
plt.xlabel("Time [s]")
plt.ylabel("Position [m]")
plt.title("Filter 2: Lift position with inferred velocity")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
plt.plot(p_y_true, p_z_true, "k--", label="actual route")
plt.plot(x_pred_2[:, 0], x_pred_2[:, 1], label="predicted route")
plt.plot(x_est_2[:, 0], x_est_2[:, 1], label="estimated route")

plt.xlabel("Horizontal position [m]")
plt.ylabel("Lift position [m]")
plt.title("Filter 2: Route comparison with inferred velocity")
plt.legend()
plt.grid(True)
plt.axis("equal")
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(t, p_y_true, "k--", label="actual horizontal position")
plt.plot(t, x_est_1[:, 0], label="Filter 1 estimate")
plt.plot(t, x_est_2[:, 0], label="Filter 2 estimate")
plt.xlabel("Time [s]")
plt.ylabel("Horizontal position [m]")
plt.title("Horizontal Position Comparison: Filter 1 vs Filter 2")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(t, p_z_true, "k--", label="actual lift position")
plt.plot(t, x_est_1[:, 1], label="Filter 1 estimate")
plt.plot(t, x_est_2[:, 1], label="Filter 2 estimate")
plt.xlabel("Time [s]")
plt.ylabel("Lift position [m]")
plt.title("Lift Position Comparison: Filter 1 vs Filter 2")
plt.legend()
plt.grid(True)
plt.show()

### 9.1 Filter 2 Results and Discussion

The second Kalman filter produced a much better result than the accelerometer-only filter. The estimated horizontal position and lift position follow the actual route very closely.

This improvement occurs because the inferred velocity signal was added as an extra measurement. Unlike the accelerometer signal, the inferred velocity was calculated from the known route distance and the measured time intervals. It therefore contains strong information about the actual movement of the phone.

The result should not be interpreted as the accelerometer alone producing a perfect position estimate. Instead, it shows that the Kalman filter can correct the accelerometer-based prediction when additional route-based velocity information is available.

The predicted and estimated values are very close because the inferred velocity measurement strongly constrains the motion of the filter. During the stationary sections, the inferred velocity is zero, which prevents position drift. During the walking and lift sections, the inferred velocity guides the filter in the correct direction.

## 10. Kalman Gain Analysis

The Kalman gain shows how much the filter trusts the measurement compared to the model prediction.

A larger Kalman gain means that the filter gives more weight to the measurement. A smaller Kalman gain means that the filter gives more weight to the model prediction.

For the first filter, the measurements are based on double-integrated acceleration, so they are noisy and affected by drift.

For the second filter, the measurements are based on the inferred velocity from the known route distance and elapsed time. This gives the filter useful additional information about the motion.

By plotting the Kalman gain, the behaviour and convergence of the filter can be analysed.

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(t, K1[:, 0, 0],"-", label="K for horizontal position measurement")
plt.plot(t, K1[:, 1, 1],"--", label="K for lift position measurement")

plt.xlabel("Time [s]")
plt.ylabel("Kalman gain")
plt.title("Kalman Gain Convergence: Filter 1")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(t, K2[:, 0, 0], label="K: horizontal position from horizontal velocity")
plt.plot(t, K2[:, 1, 1],"--", label="K: lift position from lift velocity")
plt.plot(t, K2[:, 2, 0], label="K: horizontal velocity")
plt.plot(t, K2[:, 3, 1],"--",label="K: lift velocity")

plt.xlabel("Time [s]")
plt.ylabel("Kalman gain")
plt.title("Kalman Gain Convergence: Filter 2")
plt.legend()
plt.grid(True)
plt.show()

### 10.1 Kalman Gain Discussion

The Kalman gain plots show how the filters balance the model prediction and the measurement correction.

For Filter 1, the Kalman gain starts with a short transient and then settles to an approximately constant value. The horizontal and lift position gains are almost identical because the same measurement uncertainty was used for both position measurements. The steady value of the gain shows that the filter reaches a stable balance between the accelerometer-based prediction and the noisy double-integrated position measurement.

For Filter 2, the velocity gain is much larger than the position-related gain. This is expected because the measurement vector only contains inferred velocity values. The filter therefore directly corrects the velocity states, while the position states are corrected indirectly through the motion model.

The Kalman gain settles quickly in both filters. This indicates that the filters reach a stable operating condition and do not become unstable during the journey.

## 11. Performance Comparison

The performance of the two Kalman filters is compared using the root mean square error between the estimated position and the actual route.

The RMSE is calculated as:

$$
RMSE = \sqrt{\frac{1}{N}\sum_{k=1}^{N}(x_k - \hat{x}_k)^2}
$$

A smaller RMSE means that the estimated position is closer to the actual position.

The final position error is also calculated to compare how close each filter is to the final measured route distance.


In [ ]:
def rmse(actual, estimated):
 return np.sqrt(np.mean((actual - estimated)**2))

results = pd.DataFrame({
 "Case": [
 "Filter 1: accelerometer only",
 "Filter 2: accelerometer + inferred velocity"
 ],
 "Horizontal RMSE [m]": [
 rmse(p_y_true, x_est_1[:, 0]),
 rmse(p_y_true, x_est_2[:, 0])
 ],
 "Lift RMSE [m]": [
 rmse(p_z_true, x_est_1[:, 1]),
 rmse(p_z_true, x_est_2[:, 1])
 ],
 "Final horizontal error [m]": [
 x_est_1[-1, 0] - p_y_true[-1],
 x_est_2[-1, 0] - p_y_true[-1]
 ],
 "Final lift error [m]": [
 x_est_1[-1, 1] - p_z_true[-1],
 x_est_2[-1, 1] - p_z_true[-1]
 ]
})

results

In [ ]:
plt.figure(figsize=(5, 5))
plt.bar(results["Case"], results["Horizontal RMSE [m]"])
plt.ylabel("RMSE [m]")
plt.title("Horizontal Position RMSE Comparison")
plt.xticks(rotation=15, ha="right")
plt.grid(True)
plt.show()

plt.figure(figsize=(5, 5))
plt.bar(results["Case"], results["Lift RMSE [m]"])
plt.ylabel("RMSE [m]")
plt.title("Lift Position RMSE Comparison")
plt.xticks(rotation=15, ha="right")
plt.grid(True)
plt.show()

### 11.1 Performance Comparison Discussion

The RMSE results show that Filter 2 performs significantly better than Filter 1.

Filter 1, which only uses the accelerometer signal, has a horizontal RMSE of approximately 0.515 m and a lift RMSE of approximately 1.789 m. The larger lift error is caused by drift in the z-axis acceleration signal. This drift causes the filter to estimate lift movement before the actual lift stage begins.

Filter 2 has a much smaller RMSE for both the horizontal and lift positions. This is because the inferred velocity signal is added as an extra measurement. The inferred velocity was calculated from the known route distance and the measured time intervals, so it provides strong information about the actual motion.

The very small RMSE values for Filter 2 should not be interpreted as the accelerometer alone estimating the route perfectly. Instead, the result shows that the Kalman filter can correct accelerometer drift very effectively when additional route-based velocity information is available.

The final position errors are also very small for Filter 2. This confirms that the added velocity measurement helps the filter reach the correct final horizontal and lift positions.

## 12. Final Discussion

The raw accelerometer signal contained both useful motion information and unwanted disturbances. The walking section was visible from approximately 0 s to 13 s and contained large oscillations caused by step impacts, hand movement and phone vibration. The lift section was visible from approximately 19 s to 29 s, with acceleration changes at the start and stop of the lift.

Preprocessing was required before applying the Kalman filter. The data was resampled to a constant sampling frequency of 20 Hz, the accelerometer bias was removed using the stationary calibration section from 14 s to 18 s, and a low-pass Butterworth filter was applied to reduce high-frequency disturbances.

The Kalman filter model was based on a constant acceleration motion model. This model was chosen because the phone provides acceleration measurements. The state vector included position and velocity in the horizontal walking direction and the lift direction.

The first Kalman filter used only the accelerometer signal. The results showed that this approach is affected by drift, especially in the lift direction. This is because position is obtained by double integration of acceleration, so small acceleration errors accumulate over time.

The second Kalman filter included an inferred velocity signal. This greatly improved the result because the velocity was inferred from the known route distance and elapsed time. The inferred velocity gave the filter extra information about when movement should occur and when the phone should be stationary.

The Kalman gain plots showed that both filters reached stable gain values. Filter 1 balanced the prediction with noisy position measurements obtained from double-integrated acceleration. Filter 2 placed stronger correction on the velocity states because the measurement vector contained inferred velocity values.

Overall, the results show that accelerometer-only position estimation is unreliable for accurate tracking, but the Kalman filter performs much better when additional velocity information is included.

## 13. Conclusion

A Kalman filter was implemented to estimate the position of a mobile phone using accelerometer measurements.

The accelerometer signal was first analysed to identify the walking, stationary and lift sections. The signal was then preprocessed using resampling, bias removal and low-pass filtering. These steps improved the signal and reduced the effect of noise and bias.

Two Kalman filters were implemented. The first filter used only the accelerometer signal and showed noticeable drift, especially during the lift position estimate. This demonstrated the limitation of using mobile phone accelerometer data alone for position estimation.

The second filter included an inferred velocity signal calculated from the known route distance and elapsed time. This significantly improved the estimate and produced a much smaller RMSE. The improvement occurred because the inferred velocity provided additional information that helped correct the accelerometer-based prediction.

The results show that a Kalman filter can improve position estimation, but the quality of the estimate depends strongly on the quality of the measurements used. Accelerometer-only tracking is sensitive to drift, while adding route-based velocity information produces a much more accurate and stable result.